# Critical Infrastructure Proximity and Accessibility Risk

## Objective
Analyze spatial proximity between flood hotspots and critical civic assets (hospitals, fire stations, transit corridors).

## Input Datasets
- data/staging/staging_hotspots.csv

---

### Step 1: Facility Buffer and Ingress Risk Computation

In [2]:
import os
import pandas as pd
import numpy as np

base_dir = os.path.abspath(os.path.join(os.getcwd(), "..")) if "notebooks" in os.getcwd() else os.getcwd()
df_hot = pd.read_csv(os.path.join(base_dir, "data", "staging", "staging_hotspots.csv"))

critical_facilities = [
    {"name": "KEM Hospital, Parel", "type": "Hospital", "lat": 19.0028, "lon": 72.8424},
    {"name": "Sion LTMGH Hospital", "type": "Hospital", "lat": 19.0376, "lon": 72.8601},
    {"name": "Cooper Hospital, Juhu", "type": "Hospital", "lat": 19.1082, "lon": 72.8361},
    {"name": "Kurla Bhabha Hospital", "type": "Hospital", "lat": 19.0664, "lon": 72.8835},
    {"name": "Dadar Fire Station", "type": "Fire Station", "lat": 19.0195, "lon": 72.8431},
    {"name": "Andheri Fire Station", "type": "Fire Station", "lat": 19.1170, "lon": 72.8480}
]

proximity_records = []
for fac in critical_facilities:
    dists_km = []
    for _, h in df_hot.iterrows():
        dlat = (fac["lat"] - h["latitude"]) * 111.0
        dlon = (fac["lon"] - h["longitude"]) * 111.0 * np.cos(np.radians(fac["lat"]))
        d = np.sqrt(dlat**2 + dlon**2)
        dists_km.append((d, h["name"]))
    dists_km.sort()
    closest_d, closest_spot = dists_km[0]
    proximity_records.append({
        "facility_name": fac["name"],
        "facility_type": fac["type"],
        "closest_hotspot": closest_spot,
        "distance_km": round(closest_d, 2),
        "accessibility_threat": "High" if closest_d < 1.0 else ("Moderate" if closest_d < 2.0 else "Low")
    })

print("Critical Infrastructure Proximity Audit:")
print(pd.DataFrame(proximity_records))


Critical Infrastructure Proximity Audit:
           facility_name facility_type  ... distance_km  accessibility_threat
0    KEM Hospital, Parel      Hospital  ...        1.76              Moderate
1    Sion LTMGH Hospital      Hospital  ...        0.36                  High
2  Cooper Hospital, Juhu      Hospital  ...        1.67              Moderate
3  Kurla Bhabha Hospital      Hospital  ...        0.41                  High
4     Dadar Fire Station  Fire Station  ...        0.53                  High
5   Andheri Fire Station  Fire Station  ...        0.34                  High

[6 rows x 5 columns]


---
## Analytical Limitations & Assumptions
Uses Euclidean/Haversine buffer proximity; actual vehicular route disruption depends on municipal road elevation profiles.

## Key Conclusions
Hospitals near Hindmata (KEM Parel) and Kurla West (Bhabha Hospital) face highest ingress disruption risks during extreme spells.